In [1]:
import os, glob, re
import pandas as pd

FOLDER = "simulation_output_alpha1e5"
csv_dir = os.path.join(FOLDER, "csv")

print("Folder exists:", os.path.isdir(FOLDER))
print("csv/ exists  :", os.path.isdir(csv_dir))
print()

# List every file in the csv folder
files = sorted(os.listdir(csv_dir))
print(f"Files in {csv_dir} ({len(files)} total):")
for f in files:
    print("  ", f)

Folder exists: True
csv/ exists  : True

Files in simulation_output_alpha1e5\csv (27 total):
   results_all_angles.csv
   results_angle_0.csv
   results_angle_105.csv
   results_angle_120.csv
   results_angle_135.csv
   results_angle_15.csv
   results_angle_150.csv
   results_angle_165.csv
   results_angle_180.csv
   results_angle_30.csv
   results_angle_45.csv
   results_angle_60.csv
   results_angle_75.csv
   results_angle_90.csv
   summary_angle_0.json
   summary_angle_105.json
   summary_angle_120.json
   summary_angle_135.json
   summary_angle_15.json
   summary_angle_150.json
   summary_angle_165.json
   summary_angle_180.json
   summary_angle_30.json
   summary_angle_45.json
   summary_angle_60.json
   summary_angle_75.json
   summary_angle_90.json


In [2]:
# ============================================================
# ROBUST REBUILD — handles int and float angle filenames
# ============================================================
import os, glob, re
import pandas as pd

# ============================================================
# CONFIGURAÇÃO
# ============================================================
FOLDERS = "all"    # "all"  OR  ["simulation_output_alpha1e5"]  OR  a single string

# This regex accepts: results_angle_0.csv, results_angle_0.0.csv,
# results_angle_15.5.csv, results_angle_135.csv, ...
ANGLE_RE = re.compile(r"results_angle_(\d+(?:\.\d+)?)\.csv$")


def rebuild_folder(folder, verbose=True):
    csv_dir = os.path.join(folder, "csv")
    if not os.path.isdir(csv_dir):
        if verbose:
            print(f"[skip] no csv/ in {folder}")
        return None

    all_files = glob.glob(os.path.join(csv_dir, "*.csv"))

    # Pick only the per-angle files, and parse their angle from the name
    angle_entries = []
    for p in all_files:
        m = ANGLE_RE.search(os.path.basename(p))
        if not m:
            continue
        ang = float(m.group(1))
        angle_entries.append((ang, p))

    angle_entries.sort(key=lambda t: t[0])

    if not angle_entries:
        if verbose:
            print(f"[skip] no results_angle_*.csv in {csv_dir}")
            print(f"       (files present: {[os.path.basename(f) for f in all_files]})")
        return None

    # Read each per-angle CSV
    frames = []
    angles_found = []
    for ang, path in angle_entries:
        d = pd.read_csv(path)
        if "angle_deg" not in d.columns:
            d["angle_deg"] = ang
        frames.append(d)
        angles_found.append(ang)

    combined = pd.concat(frames, ignore_index=True)

    # Canonical column order (only those present)
    cols = ["alpha", "angle_deg", "status",
            "x0", "y0", "z0", "xf", "yf", "zf", "gamma"]
    have = [c for c in cols if c in combined.columns]
    combined = combined[have]

    # Sort by alpha, then angle, then status (nice for downstream plotting)
    sort_cols = [c for c in ["alpha", "angle_deg", "status"] if c in combined.columns]
    if sort_cols:
        combined = combined.sort_values(sort_cols).reset_index(drop=True)

    out = os.path.join(csv_dir, "results_all_angles.csv")
    combined.to_csv(out, index=False)

    if verbose:
        print(f"[OK] {folder}")
        print(f"     angles found ({len(angles_found)}): "
              f"{[int(a) if a == int(a) else a for a in angles_found]}")
        print(f"     rows total  : {len(combined)}")
        for ang in angles_found:
            n = (combined.angle_deg == ang).sum()
            print(f"        {int(ang) if ang==int(ang) else ang:>6}° : {n}")
        print(f"     wrote: {out}")
    return combined


# ============================================================
# RUN
# ============================================================
if FOLDERS == "all":
    folders = sorted(glob.glob("simulation_output_*"))
else:
    folders = [FOLDERS] if isinstance(FOLDERS, str) else list(FOLDERS)

for folder in folders:
    if not os.path.isdir(folder):
        print(f"[skip] not a folder: {folder}")
        continue
    rebuild_folder(folder)
    print()

[OK] simulation_output_alpha1e2
     angles found (13): [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180]
     rows total  : 13000
             0° : 1000
            15° : 1000
            30° : 1000
            45° : 1000
            60° : 1000
            75° : 1000
            90° : 1000
           105° : 1000
           120° : 1000
           135° : 1000
           150° : 1000
           165° : 1000
           180° : 1000
     wrote: simulation_output_alpha1e2\csv\results_all_angles.csv

[OK] simulation_output_alpha1e3
     angles found (13): [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180]
     rows total  : 13000
             0° : 1000
            15° : 1000
            30° : 1000
            45° : 1000
            60° : 1000
            75° : 1000
            90° : 1000
           105° : 1000
           120° : 1000
           135° : 1000
           150° : 1000
           165° : 1000
           180° : 1000
     wrote: simulation_output_alpha1e3\csv\results_all_an

In [3]:
import glob, os
import pandas as pd

all_combined = []
for folder in sorted(glob.glob("simulation_output_*")):
    path = os.path.join(folder, "csv", "results_all_angles.csv")
    if not os.path.isfile(path):
        print(f"[skip] {path}")
        continue
    d = pd.read_csv(path)
    d["source_folder"] = os.path.basename(folder)
    all_combined.append(d)
    print(f"[add] {path}  ->  {len(d)} rows")

master = pd.concat(all_combined, ignore_index=True)
master.to_csv("particles_all_runs.csv", index=False)
print(f"\n[OK] particles_all_runs.csv  ({len(master)} rows)\n")

# Verification grid
pivot = master.groupby(["alpha", "angle_deg"]).size().unstack(fill_value=0)
print(pivot)

[add] simulation_output_alpha1e2\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e3\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e4\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e5\csv\results_all_angles.csv  ->  13000 rows

[OK] particles_all_runs.csv  (52000 rows)

angle_deg   0     15    30    45    60    75    90    105   120   135   150  \
alpha                                                                         
100        1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
1000       1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
10000      1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
100000     1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   

angle_deg   165   180  
alpha                  
100        1000  1000  
1000       1000  1000  
10000      1000  1000  
100000     1000  1000  


In [4]:
import glob, os
import pandas as pd

all_combined = []
for folder in sorted(glob.glob("simulation_output_*")):
    path = os.path.join(folder, "csv", "results_all_angles.csv")
    if not os.path.isfile(path):
        print(f"[skip] {path}")
        continue
    d = pd.read_csv(path)
    d["source_folder"] = os.path.basename(folder)
    all_combined.append(d)
    print(f"[add] {path}  ->  {len(d)} rows")

master = pd.concat(all_combined, ignore_index=True)
master.to_csv("particles_all_runs.csv", index=False)
print(f"\n[OK] particles_all_runs.csv  ({len(master)} rows)\n")

# Verification grid
pivot = master.groupby(["alpha", "angle_deg"]).size().unstack(fill_value=0)
print(pivot)

[add] simulation_output_alpha1e2\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e3\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e4\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e5\csv\results_all_angles.csv  ->  13000 rows

[OK] particles_all_runs.csv  (52000 rows)

angle_deg   0     15    30    45    60    75    90    105   120   135   150  \
alpha                                                                         
100        1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
1000       1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
10000      1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
100000     1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   

angle_deg   165   180  
alpha                  
100        1000  1000  
1000       1000  1000  
10000      1000  1000  
100000     1000  1000  


In [5]:
# ============================================================
# MERGE ALL ALPHA FOLDERS INTO ONE MASTER CSV
# ============================================================
import glob, os
import pandas as pd

all_combined = []
for folder in sorted(glob.glob("simulation_output_*")):
    path = os.path.join(folder, "csv", "results_all_angles.csv")
    if not os.path.isfile(path):
        print(f"[skip] no combined CSV: {path}")
        continue
    d = pd.read_csv(path)
    d["source_folder"] = os.path.basename(folder)   # provenance
    all_combined.append(d)
    print(f"[add] {path}  ->  {len(d)} rows")

if all_combined:
    master = pd.concat(all_combined, ignore_index=True)
    master.to_csv("particles_all_runs.csv", index=False)
    print()
    print(f"[OK] particles_all_runs.csv  ({len(master)} rows)")
    print(master.groupby(["alpha", "angle_deg"]).size().unstack(fill_value=0))
else:
    print("No combined CSVs found.")

[add] simulation_output_alpha1e2\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e3\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e4\csv\results_all_angles.csv  ->  13000 rows
[add] simulation_output_alpha1e5\csv\results_all_angles.csv  ->  13000 rows

[OK] particles_all_runs.csv  (52000 rows)
angle_deg   0     15    30    45    60    75    90    105   120   135   150  \
alpha                                                                         
100        1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
1000       1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
10000      1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
100000     1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   

angle_deg   165   180  
alpha                  
100        1000  1000  
1000       1000  1000  
10000      1000  1000  
100000     1000  1000  


In [6]:
import pandas as pd
df = pd.read_csv("particles_all_runs.csv")
EXPECTED_ANGLES = [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180]

for a, g in df.groupby("alpha"):
    have = sorted(g.angle_deg.unique())
    missing = [x for x in EXPECTED_ANGLES if x not in have]
    tag = "OK" if not missing else f"MISSING {missing}"
    print(f"alpha = {a:>10.0f}  angles: {have}  [{tag}]")

alpha =        100  angles: [np.int64(0), np.int64(15), np.int64(30), np.int64(45), np.int64(60), np.int64(75), np.int64(90), np.int64(105), np.int64(120), np.int64(135), np.int64(150), np.int64(165), np.int64(180)]  [OK]
alpha =       1000  angles: [np.int64(0), np.int64(15), np.int64(30), np.int64(45), np.int64(60), np.int64(75), np.int64(90), np.int64(105), np.int64(120), np.int64(135), np.int64(150), np.int64(165), np.int64(180)]  [OK]
alpha =      10000  angles: [np.int64(0), np.int64(15), np.int64(30), np.int64(45), np.int64(60), np.int64(75), np.int64(90), np.int64(105), np.int64(120), np.int64(135), np.int64(150), np.int64(165), np.int64(180)]  [OK]
alpha =     100000  angles: [np.int64(0), np.int64(15), np.int64(30), np.int64(45), np.int64(60), np.int64(75), np.int64(90), np.int64(105), np.int64(120), np.int64(135), np.int64(150), np.int64(165), np.int64(180)]  [OK]
